# Baseline vs ControlNet Experiment

This notebook compares Stable Diffusion img2img against Stable Diffusion + ControlNet depth.

## Aim

This experiment compares standard Stable Diffusion
with Stable Diffusion + ControlNet depth guidance.

The goal is to evaluate whether Control Net guidance, in thsi case based on MiDas Depth map conditioning improves:

- structural consistency
- preservation of important objects
- semantic accuracy of transformation
- overall visual realism

for conditional image translation tasks such as season translation

In [ ]:
!pip -q install -r /content/image-data-generation/requirements.txt

In [ ]:
import os
import time
import gc
import torch
import numpy as np
import cv2
from PIL import Image
from IPython.display import display
import matplotlib.pyplot as plt


dataset_dir = "/content/image-data-generation/dataset/source_road_scenes"
results_dir = "/content/image-data-generation/results/baseline_vs_controlnet"

os.makedirs(results_dir, exist_ok=True)

image_paths = sorted([
    os.path.join(dataset_dir, f)
    for f in os.listdir(dataset_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".jfif"))
])


In [ ]:
if len(image_paths) == 0:
    raise FileNotFoundError("No road-scene images found in the dataset folder.")

print(f"Found {len(image_paths)} road-scene images.")

user_prompt = "Convert image to snowy winter with heavy snowfall"

prompt = (
    f"{user_prompt}. "
    "Preserve original road layout, vehicles, object positions, and perspective."
)

negative_prompt = (
    "blurry, distorted cars, distorted road, bad geometry, low quality, warped perspective"
)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline
from diffusers import ControlNetModel, StableDiffusionControlNetImg2ImgPipeline

print("Loading baseline Stable Diffusion...")
baseline_pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "sd-legacy/stable-diffusion-v1-5",
    torch_dtype=dtype,
    safety_checker=None
).to(device)

baseline_pipe.enable_attention_slicing()

print("Loading MiDaS...")
midas_device = torch.device(device)

midas_model = torch.hub.load("intel-isl/MiDaS", "DPT_Hybrid")
midas_model.to(midas_device).eval()

midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
transform = midas_transforms.dpt_transform

print("Loading ControlNet...")
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-depth",
    torch_dtype=dtype
)

print("Loading Stable Diffusion + ControlNet...")
control_pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    "sd-legacy/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=dtype,
    safety_checker=None
).to(device)

control_pipe.enable_attention_slicing()

In [ ]:
for image_path in image_paths:
    image_id = os.path.splitext(os.path.basename(image_path))[0]
    output_dir = os.path.join(results_dir, image_id)
    os.makedirs(output_dir, exist_ok=True)

    print("\nProcessing:", image_id, image_path)

    image = Image.open(image_path).convert("RGB")
    init_image = image.resize((512, 512))

    # Save original
    init_image.save(os.path.join(output_dir, "original.png"))

    # Baseline SD img2img
    generator = torch.Generator(device=device).manual_seed(42)

    start_time = time.time()
    baseline_result = baseline_pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=init_image,
        strength=0.8,
        guidance_scale=7.5,
        num_inference_steps=30,
        generator=generator
    ).images[0]
    baseline_time = time.time() - start_time

    baseline_result.save(os.path.join(output_dir, "baseline_sd.png"))

    # MiDaS
    img_np = np.array(image)
    input_batch = transform(img_np).to(midas_device)

    with torch.no_grad():
        prediction = midas_model(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=img_np.shape[:2],
            mode="bicubic",
            align_corners=False
        ).squeeze(0).squeeze(0)

    depth = prediction.detach().cpu().numpy()

    depth_vis = cv2.normalize(depth, None, 0, 255, norm_type=cv2.NORM_MINMAX)
    depth_vis = depth_vis.astype(np.uint8)

    depth_gray = Image.fromarray(depth_vis).convert("RGB").resize((512, 512))
    depth_gray.save(os.path.join(output_dir, "depth_gray.png"))

    # ControlNet Stable Diffusion
    generator = torch.Generator(device=device).manual_seed(42)

    start_time = time.time()
    controlnet_result = control_pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=init_image,
        control_image=depth_gray,
        strength=0.8,
        guidance_scale=7.5,
        controlnet_conditioning_scale=1.0,
        num_inference_steps=30,
        generator=generator
    ).images[0]
    controlnet_time = time.time() - start_time

    controlnet_result.save(os.path.join(output_dir, "controlnet_depth.png"))

    # Comparison figure
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    axes[0].imshow(init_image)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(baseline_result)
    axes[1].set_title("Stable Diffusion Only")
    axes[1].axis("off")

    axes[2].imshow(controlnet_result)
    axes[2].set_title("SD + ControlNet Depth")
    axes[2].axis("off")

    plt.tight_layout()
    comparison_path = os.path.join(output_dir, "comparison.png")
    plt.savefig(comparison_path, dpi=300)
    plt.show()
    plt.close()

    # Save metadata to support traceability
    with open(os.path.join(output_dir, "metadata.txt"), "w") as f:
        f.write(f"Image ID: {image_id}\n")
        f.write(f"Input path: {image_path}\n")
        f.write(f"Prompt: {prompt}\n")
        f.write(f"Negative prompt: {negative_prompt}\n")
        f.write("Seed: 42\n")
        f.write("Image size: 512x512\n")
        f.write("Strength: 0.8\n")
        f.write("Guidance scale: 7.5\n")
        f.write("Inference steps: 30\n")
        f.write("ControlNet conditioning scale: 1.0\n")
        f.write(f"Baseline runtime: {baseline_time:.2f} seconds\n")
        f.write(f"ControlNet runtime: {controlnet_time:.2f} seconds\n")

    print(f"Saved results to {output_dir}")

    expected_files = [
        "original.png",
        "baseline_sd.png",
        "depth_gray.png",
        "controlnet_depth.png",
        "comparison.png",
        "metadata.txt"
    ]

    missing_files = [
        file for file in expected_files
        if not os.path.exists(os.path.join(output_dir, file))
    ]

    if missing_files:
        print("Missing files:", missing_files)
    else:
        print("All expected files saved for", image_id)

    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
print("All images processed.")

del baseline_pipe
del control_pipe
del controlnet
del midas_model
del midas_transforms
del transform

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared.")